# Model report
Saved outputs only. No inference, threshold fitting, or HTML export. Fill in the plain-language notes; do not infer a gain from a changed test protocol.

In [ ]:
ROOT_HINT = None
RUN = None
EVALUATION_ID = None
REFERENCE_RUN = None
REFERENCE_EVALUATION_ID = None

In [ ]:
import os, sys
from pathlib import Path
os.environ.update(CUDA_VISIBLE_DEVICES='', HIP_VISIBLE_DEVICES='', MPLBACKEND='Agg')
candidates = ([Path(ROOT_HINT)] if ROOT_HINT else []) + [Path.cwd(), *Path.cwd().parents]
ROOT = next(p for p in candidates if (p/'cvdl/reporting.py').is_file())
sys.path.insert(0, str(ROOT))
from cvdl.reporting import SavedReport, cpu_offline_guard, compatible, show_table, training_plots, miss_plots, hard_plots
cpu_offline_guard()
from IPython.display import display, Markdown
report = SavedReport(ROOT, RUN, EVALUATION_ID)
reference = SavedReport(ROOT, REFERENCE_RUN, REFERENCE_EVALUATION_ID) if REFERENCE_RUN else None
if reference: compatible(report, reference)
assert 'torch' not in sys.modules


## 1. Run card
One saved validation threshold is used for every deployment table and hard-test cell. Post-hoc calibration does not alter the training checkpoint or its original validation curves.

In [ ]:
show_table(report.run_card())
print("Validation cutoff (full precision):", repr(report.calibration["cutoff"]))
print("Threshold ID:", report.calibration["threshold_id"])

## 2. Original training curves
Dashed lines mark drops in the saved learning-rate trace. These are the original checkpoint-selection validation curves, **not** evaluation calibration curves. The original training plot is embedded below.

In [ ]:
training_plots(report)

## 3. Headline deployment results
FA, miss and P_E use the ONE saved validation cutoff. AUC does not use a cutoff. Rates are shown as percentages; AUC is 0–1. Single-source panels have identical overall and source rows.

In [ ]:
headline_table = report.headline
show_table(headline_table)

### Diagnostics only — not deployment results
`test_min_P_E` is optimised separately on each test subset and is optimistic. `*_at_0_5` uses fixed 0.5, not the validation cutoff. Do not call either a validation-calibrated result.

In [ ]:
show_table(report.diagnostics.loc[report.diagnostics.category.isin(["overall", "source"])])

## 4. Misses by k, requested bpp and k × bpp
Computed from saved stego probabilities at the saved cutoff only. Bins: [0.05,0.10), [0.10,0.15), [0.15,0.35), [0.35,0.50]. No new threshold is fitted.

In [ ]:
miss_tables = miss_plots(report)

## 5. Hard-test FA and miss
Each cell uses the same validation cutoff. Error bars are saved Wilson 95% ranges (tile/pair trials, not photo-cluster intervals). High noisy-clean FA can reduce miss without improving separation. Cells overlap: do not treat them as independent experiments.

In [ ]:
hard_plots(report)
if report.hard is not None:
    show_table(report.hard[["panel","k","condition","pairs","distinct_clean_tiles","FA","FA_ci95_low","FA_ci95_high","miss","miss_ci95_low","miss_ci95_high","P_E_at_validation_cutoff","AUC"]])

## 6. Optional reference model
Only matching frozen test recipes can be compared. Each model keeps its OWN validation threshold. The reference is not used to choose a threshold or a checkpoint.

In [ ]:
if reference is None:
    print('No reference model supplied; comparison skipped.')
else:
    display(Markdown(f'**Primary:** {report.run.name} — `{report.calibration["threshold_origin"]}`, cutoff `{report.calibration["cutoff"]!r}`. **Reference:** {reference.run.name} — `{reference.calibration["threshold_origin"]}`, cutoff `{reference.calibration["cutoff"]!r}`. Each model keeps its own validation threshold.'))
    display(Markdown('### Reference run card'))
    show_table(reference.run_card())
    display(Markdown('### Primary and reference headline tables, side by side'))
    keys = ['panel','category','value']
    show_table(report.headline.merge(reference.headline, on=keys, suffixes=('_primary','_reference'), validate='one_to_one'))
    hard_plots(report, reference)
    display(Markdown('### Reference original training curves'))
    training_plots(reference)
    display(Markdown('### Reference misses at its own saved cutoff'))
    reference_miss_tables = miss_plots(reference)
    for name, keys in [('k', ['panel','k']), ('bpp', ['panel','bpp_bin']), ('k_bpp', ['panel','k','bpp_bin'])]:
        display(Markdown(f'### Side-by-side miss table: {name}'))
        show_table(miss_tables[name].merge(reference_miss_tables[name], on=keys, suffixes=('_primary','_reference'), validate='one_to_one'))


## 7. Higher-k check (optional)
If scored, k5–8 uses the same selected tiles and the same saved cutoff. It does not set a new threshold.

In [ ]:
if report.higher is None:
    print('Higher-k output not requested; skipped.')
else:
    show_table(report.higher)
    if reference is not None and reference.higher is not None:
        show_table(reference.higher)


## 8. If the source were known — REFERENCE ONLY
These source-specific cutoffs are not the deployment policy. No DIV2K reference exists. ALASKA uses one reference for seen and unseen. Mixed RAISE+DIV2K reference aggregates are deliberately omitted.

In [ ]:
show_table(report.reference_thresholds)
show_table(report.source_reference)
if reference is not None:
    display(Markdown("### Reference-model source-known references"))
    show_table(reference.reference_thresholds)

## 9. Author notes — fill these in
- **What did this run change?** [Pending: author]
- **What do the saved results show?** [Pending: cite rows/cells, including FA as well as miss]
- **What remains weak?** [Pending: author]
- **How does the optional reference compare?** [Pending / not supplied]
- **Limits:** [Pending: source mix, geometry, small differences, correlated tiles, and test-protocol changes]
- **Next decision:** [Pending: author]

No claims are generated automatically. Old 512/k1–8 results are not like-for-like with the balanced 256/k1–4 suite.

In [ ]:
assert os.environ['CUDA_VISIBLE_DEVICES'] == '' and os.environ['HIP_VISIBLE_DEVICES'] == ''
assert 'torch' not in sys.modules
import socket
try:
    socket.getaddrinfo('external-network.invalid', 443)
except PermissionError:
    print('Verified: CPU-only report, inference imports blocked, external network disabled; saved data only.')
else:
    raise AssertionError('external network guard missing')
